# Bayesian Last Layer + BoTorch: Hafif Belirsizlik-Aware Bayesçi Optimizasyon

Bu notebook, **tam BNN yerine yalnız son katmanı Bayesçi** yapan bir `Neural Linear / Bayesian Last Layer (BLL)` surrogate kurar ve bunu BoTorch ile pahalı bir proses fonksiyonunun optimizasyonunda kullanır.

Akış:

```text
başlangıç deneyleri
      ↓
deterministik sinir ağı ile özellik öğrenimi
      ↓
φ(x): son gizli katman temsili
      ↓
Bayesçi lineer son katman
β | D ~ N(m, Σ)
      ↓
posterior ağırlık örnekleri
      ↓
BoTorch EnsembleModel
      ↓
qLogExpectedImprovement
      ↓
yeni deney noktası
```

Bu model **tam BNN değildir**. Gizli katman ağırlıkları nokta tahminidir; yalnız çıktı katmanındaki ağırlıklar Bayesçidir. Avantajı, full BNN'ye göre posterior güncellemesinin çok daha ucuz olmasıdır.


In [ ]:
# Gerekirse:
# %pip install torch botorch numpy pandas matplotlib

import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from botorch.models.ensemble import EnsembleModel
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.optim import optimize_acqf

torch.manual_seed(42)
np.random.seed(42)
torch.set_default_dtype(torch.double)


## 1. Pahalı proses fonksiyonu

İki proses parametresini optimize ettiğimizi düşünelim:

- \(x_1\): normalize edilmiş sıcaklık / set-point,
- \(x_2\): normalize edilmiş hız / basınç.

Amaç kalite/verim skorunu **maksimize etmek**.

Gerçek uygulamada `black_box(X)` yerine fiziksel deney, ayrık olay simülasyonu, FEA/CFD veya dijital ikiz çağrısı gelir.


In [ ]:
def black_box(X):
    x1 = X[..., 0]
    x2 = X[..., 1]
    peak1 = 1.30 * torch.exp(-22.0 * ((x1 - 0.72)**2 + (x2 - 0.30)**2))
    peak2 = 0.75 * torch.exp(-35.0 * ((x1 - 0.28)**2 + (x2 - 0.78)**2))
    ripple = 0.12 * torch.sin(8*x1) * torch.cos(7*x2)
    penalty = 0.10 * (x1 - x2)**2
    return peak1 + peak2 + ripple - penalty

bounds = torch.tensor([[0.0, 0.0], [1.0, 1.0]])

sobol = torch.quasirandom.SobolEngine(dimension=2, scramble=True, seed=42)
train_X = sobol.draw(16)
train_Y = black_box(train_X).unsqueeze(-1)

print("Başlangıç en iyi skor:", float(train_Y.max()))


In [ ]:
grid_n = 80
g1 = torch.linspace(0, 1, grid_n)
g2 = torch.linspace(0, 1, grid_n)
G1, G2 = torch.meshgrid(g1, g2, indexing="ij")
grid_X = torch.stack([G1.reshape(-1), G2.reshape(-1)], dim=-1)
grid_Y = black_box(grid_X).reshape(grid_n, grid_n)

plt.figure(figsize=(6,5))
plt.contourf(G1.numpy(), G2.numpy(), grid_Y.numpy(), levels=30)
plt.scatter(train_X[:,0], train_X[:,1], marker="x", label="Başlangıç deneyleri")
plt.xlabel("x1")
plt.ylabel("x2")
plt.legend()
plt.title("Gerçek black-box yüzeyi (öğretim için biliniyor)")
plt.show()


## 2. Deterministik özellik çıkarıcı

Sinir ağının gizli katmanları deterministik olarak eğitilir. Daha sonra son gizli katman çıktısı

\[
\phi(x)\in\mathbb{R}^h
\]

Bayesçi lineer regresyonun girdisi olur.

Bu, full BNN'den temel farktır: temsil öğreniminin belirsizliğini modellemiyoruz.


In [ ]:
class FeatureNet(nn.Module):
    def __init__(self, d=2, h1=32, h2=16):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(d, h1),
            nn.Tanh(),
            nn.Linear(h1, h2),
            nn.Tanh(),
        )
        self.head = nn.Linear(h2, 1)

    def features(self, X):
        return self.backbone(X)

    def forward(self, X):
        return self.head(self.features(X))


def fit_feature_net(X, Y, steps=1200, lr=0.01):
    net = FeatureNet(d=X.shape[-1])
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    y_mean = Y.mean()
    y_std = Y.std().clamp_min(1e-6)
    Yz = (Y - y_mean) / y_std

    for step in range(steps):
        opt.zero_grad()
        pred = net(X)
        loss = ((pred - Yz)**2).mean()
        loss.backward()
        opt.step()

    return net, y_mean.detach(), y_std.detach()

feature_net, y_mean, y_std = fit_feature_net(train_X, train_Y)

with torch.no_grad():
    pred = feature_net(train_X) * y_std + y_mean
    rmse = torch.sqrt(((pred - train_Y)**2).mean())

print("Eğitim RMSE:", float(rmse))


## 3. Son katmanda analitik Bayesçi lineer regresyon

Özellik vektörüne bias ekleyelim:

\[
\tilde\phi(x)=[1,\phi(x)].
\]

Son katman:

\[
f(x)=\tilde\phi(x)^\top \beta
\]

ve önsel:

\[
\beta\sim\mathcal N(0,\alpha^{-1}I).
\]

Gaussian gözlem modeli altında posterior kapalı formdadır:

\[
\Sigma =
\left(
\alpha I+\tau\Phi^\top\Phi
\right)^{-1},
\]

\[
m=
\tau\Sigma\Phi^\top y,
\]

burada \(\tau=1/\sigma_\epsilon^2\) gürültü precision'ıdır.

Bu analitik güncelleme, BLL'nin full BNN'ye göre ucuz olmasının ana nedenlerinden biridir.


In [ ]:
def design_matrix(net, X):
    phi = net.features(X)
    ones = torch.ones(*phi.shape[:-1], 1, dtype=phi.dtype, device=phi.device)
    return torch.cat([ones, phi], dim=-1)


def fit_bayesian_last_layer(net, X, Y, y_mean, y_std, prior_precision=1.0):
    Yz = ((Y - y_mean) / y_std).squeeze(-1)

    with torch.no_grad():
        Phi = design_matrix(net, X)
        deterministic_resid = net(X).squeeze(-1) - Yz
        noise_var = deterministic_resid.pow(2).mean().clamp_min(2e-3)

        p = Phi.shape[-1]
        I = torch.eye(p, dtype=Phi.dtype)
        noise_precision = 1.0 / noise_var

        precision = prior_precision * I + noise_precision * Phi.T @ Phi
        Sigma = torch.linalg.inv(precision)
        m = noise_precision * Sigma @ Phi.T @ Yz

    return m, Sigma, noise_var


beta_mean, beta_cov, noise_var = fit_bayesian_last_layer(
    feature_net, train_X, train_Y, y_mean, y_std
)

posterior_beta = torch.distributions.MultivariateNormal(
    beta_mean, covariance_matrix=beta_cov
)
beta_samples = posterior_beta.rsample((256,))

print("Bayesçi son katman boyutu:", beta_mean.numel())
print("Yaklaşık normalize gürültü varyansı:", float(noise_var))


## 4. BoTorch için örneklenebilir surrogate

BoTorch'un Monte Carlo acquisition fonksiyonları GP zorunluluğu koymaz. Modelin `posterior()` üzerinden örneklenebilir bir posterior sağlaması yeterlidir.

`EnsembleModel`, son katman posteriorundan çektiğimiz ağırlık örneklerini bir ensemble posterior olarak kullanır. Aynı \(\beta^{(s)}\) örneği tüm aday noktalarında kullanıldığı için posterior örnekleri karar noktaları arasında ortak fonksiyon örnekleri oluşturur.

Gizli katmanlar differentiable kaldığından acquisition fonksiyonunun girdiye göre türevi alınabilir.


In [ ]:
class BayesianLastLayerEnsemble(EnsembleModel):
    _num_outputs = 1

    def __init__(self, feature_net, beta_samples, y_mean, y_std):
        super().__init__()
        self.feature_net = feature_net
        self.register_buffer("beta_samples", beta_samples)
        self.register_buffer("y_mean_buf", y_mean.reshape(()))
        self.register_buffer("y_std_buf", y_std.reshape(()))

        for p in self.feature_net.parameters():
            p.requires_grad_(False)

    def forward(self, X):
        # X: ... x q x d
        phi = self.feature_net.features(X)
        ones = torch.ones(*phi.shape[:-1], 1, dtype=phi.dtype, device=phi.device)
        Phi = torch.cat([ones, phi], dim=-1)

        # ... x q x p  ve  s x p  -> ... x s x q
        z = torch.einsum("...qp,sp->...sq", Phi, self.beta_samples)
        y = z * self.y_std_buf + self.y_mean_buf
        return y.unsqueeze(-1)  # ... x s x q x 1


bll_model = BayesianLastLayerEnsemble(
    feature_net, beta_samples, y_mean, y_std
)

test_post = bll_model.posterior(train_X[:3].unsqueeze(0))
print("Posterior mean shape:", tuple(test_post.mean.shape))
print("Posterior variance shape:", tuple(test_post.variance.shape))


## 5. Posterior belirsizliğini görselleştirme

Aşağıdaki yüzeyler BLL posterior ortalamasını ve epistemik standart sapmayı gösterir.

Dikkat: Bu belirsizlik **yalnız son katman ağırlık belirsizliğidir**. Özellik çıkarıcının ağırlık belirsizliği hesaba katılmaz. Bu nedenle BLL, full BNN'nin hesaplama açısından ucuz bir yaklaşımıdır; onunla özdeş değildir.


In [ ]:
with torch.no_grad():
    post = bll_model.posterior(grid_X.unsqueeze(-2))
    post_mean = post.mean.squeeze(-1).squeeze(-1).reshape(grid_n, grid_n)
    post_std = post.variance.sqrt().squeeze(-1).squeeze(-1).reshape(grid_n, grid_n)

plt.figure(figsize=(6,5))
plt.contourf(G1.numpy(), G2.numpy(), post_mean.numpy(), levels=30)
plt.scatter(train_X[:,0], train_X[:,1], marker="x")
plt.title("BLL posterior ortalaması")
plt.xlabel("x1"); plt.ylabel("x2")
plt.show()

plt.figure(figsize=(6,5))
plt.contourf(G1.numpy(), G2.numpy(), post_std.numpy(), levels=30)
plt.scatter(train_X[:,0], train_X[:,1], marker="x")
plt.title("BLL epistemik standart sapması")
plt.xlabel("x1"); plt.ylabel("x2")
plt.show()


## 6. BoTorch acquisition: qLogExpectedImprovement

Şimdi BLL posteriorunu exploration/exploitation kararına bağlayalım.

\[
x_{next}
=
\arg\max_x
\operatorname{qLogEI}(x).
\]

Bu adımda neural network doğrudan karar vermez. BLL posterioru belirsiz surrogate sağlar; BoTorch acquisition fonksiyonu ise hangi deneyin yapılacağını seçer.


In [ ]:
best_f = train_Y.max()

acq = qLogExpectedImprovement(
    model=bll_model,
    best_f=best_f,
)

candidate, acq_value = optimize_acqf(
    acq_function=acq,
    bounds=bounds,
    q=1,
    num_restarts=12,
    raw_samples=256,
)

candidate_y = black_box(candidate).unsqueeze(-1)

print("Önerilen yeni deney:", candidate.detach().numpy().round(4))
print("Gerçek skor:", float(candidate_y))
print("Mevcut en iyi skor:", float(best_f))
print("Acquisition değeri:", float(acq_value))


## 7. Kısa sequential optimization döngüsü

Aşağıdaki hücre birkaç ek deney boyunca modeli yeniden eğitir ve son katman posteriorunu günceller.

Bu küçük öğretim örneğinde özellik çıkarıcı her turda yeniden eğitiliyor. Gerçek online sistemde:

- backbone daha seyrek güncellenebilir,
- yalnız Bayesçi son katman sık güncellenebilir,
- replay / retraining stratejisi ayrıca tasarlanmalıdır.


In [ ]:
X_bo = train_X.clone()
Y_bo = train_Y.clone()
history = [float(Y_bo.max())]

for iteration in range(6):
    net, ym, ys = fit_feature_net(X_bo, Y_bo, steps=600, lr=0.01)
    bm, bc, nv = fit_bayesian_last_layer(net, X_bo, Y_bo, ym, ys)

    dist_beta = torch.distributions.MultivariateNormal(bm, covariance_matrix=bc)
    bs = dist_beta.rsample((192,))
    model_i = BayesianLastLayerEnsemble(net, bs, ym, ys)

    acq_i = qLogExpectedImprovement(model=model_i, best_f=Y_bo.max())
    x_new, _ = optimize_acqf(
        acq_i, bounds=bounds, q=1,
        num_restarts=8, raw_samples=128
    )
    y_new = black_box(x_new).unsqueeze(-1)

    X_bo = torch.cat([X_bo, x_new.detach()], dim=0)
    Y_bo = torch.cat([Y_bo, y_new.detach()], dim=0)
    history.append(float(Y_bo.max()))

    print(
        f"Tur {iteration+1}: x={x_new.detach().numpy().round(3)}, "
        f"y={float(y_new):.4f}, best={history[-1]:.4f}"
    )

plt.plot(range(len(history)), history, marker="o")
plt.xlabel("Ek BO turu")
plt.ylabel("Şimdiye kadarki en iyi skor")
plt.title("Bayesian Last Layer + BoTorch")
plt.show()


## 8. Full BNN ile Bayesian Last Layer farkı

| Özellik | Full BNN | Bayesian Last Layer |
|---|---|---|
| Gizli katman ağırlıkları | dağılım | nokta tahmini |
| Son katman | dağılım | dağılım |
| Posterior maliyeti | yüksek | düşük |
| Özellik belirsizliği | temsil edilebilir | temsil edilmez |
| Online güncelleme | daha zor | çok daha kolay |
| BoTorch entegrasyonu | mümkün | çok pratik |
| Büyük endüstriyel veri | maliyetli olabilir | çoğu zaman daha uygulanabilir |

BLL özellikle şu koşullarda güçlü bir pragmatik alternatiftir:

- feature representation yeterince iyi öğrenilebiliyorsa,
- sık posterior güncellemesi gerekiyorsa,
- deney/optimizasyon döngüsü online çalışıyorsa,
- full BNN inference maliyeti operasyonel olarak fazla ise.

Ama BLL'nin belirsizliği otomatik olarak iyi kalibre değildir. OOD bölgelerinde deterministik backbone aşırı güvenli temsil üretebilir. Bu nedenle GP, full BNN ve Deep Ensemble baseline'larıyla karşılaştırılmalıdır.


## 9. Matematiksel statü ve kaynaklar

Bu yapı matematiksel olarak standart iki parçanın birleşimidir:

1. sinir ağıyla öğrenilmiş özellikler \(\phi(x)\),
2. bu özellikler üzerinde Bayesçi lineer regresyon.

Gaussian prior + Gaussian likelihood altında son katman posteriorunun kapalı formda olması klasik Bayesçi lineer regresyon sonucudur.

BoTorch v0.18.1 dokümantasyonu, Monte Carlo acquisition fonksiyonlarının GP olmayan ve `posterior()/rsample()` arayüzünü sağlayan custom modellerle çalışabildiğini belirtir:

- https://botorch.org/docs/models
- https://botorch.org/docs/tutorials/custom_model

Bayesian last layer üzerine örnek literatür:

- Fiedler & Lucia (2023), *Improved uncertainty quantification for neural networks with Bayesian last layer*, arXiv:2302.10975
- Moberg et al. (2019), *Bayesian Linear Regression on Deep Representations*, arXiv:1912.06760

### Uyarı

Bu notebookta backbone aynı veri üzerinde MSE ile eğitilip sonra sabitleniyor. Bu, tam Bayesçi inference değildir ve temsil öğrenimi belirsizliğini posteriora taşımaz. Bu nedenle yöntem **hesaplama açısından pragmatik bir yaklaşık belirsizlik modeli** olarak değerlendirilmelidir.
